In [1]:
#@title LICENSE
# Licensed under the Apache License, Version 2.0

# JaxPruner X FedJAX
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dimarkov/jaxpruner/blob/main/colabs/fedjax_jaxpruner.ipynb)

This interactive colab explores pruning in federated learning. It demonstrates how to easily experiment with various pruning methods on federated training of EMNIST using FedJAX with the `jaxpruner` library.

Pruning is conducted on the server, and is easily added to federated training by
* wrapping the existing server optax optimizer with a `jaxpruner` updater
* applying a post gradient update step on the server model

### Imports

In [2]:
import collections
from ml_collections import config_dict
from typing import Any, Callable, Mapping, Sequence, Tuple

import jax
import jax.numpy as jnp
import optax

import fedjax



I0000 00:00:1791822232.060938  115080 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


I0000 00:00:1791822233.346031  115080 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
import sys
if 'google.colab' in sys.modules:
  !pip3 install git+https://github.com/dimarkov/jaxpruner
import jaxpruner

### Configure Pruning Algorithm

Here we specify the sparsity config to define the pruning algorithm to be used.

In [4]:
SPARSITY_CONFIG = config_dict.ConfigDict()
SPARSITY_CONFIG.algorithm = 'magnitude' # what pruning algorithm to apply, options: 'magnitude', 'random', 'saliency'
SPARSITY_CONFIG.dist_type = 'uniform' # how to distribute sparsity to variables, options:  str, 'erk', 'uniform'
SPARSITY_CONFIG.update_freq = 10 # how often to apply pruning
SPARSITY_CONFIG.update_start_step = 250 # when to start pruning
SPARSITY_CONFIG.update_end_step = 750 # when to stop pruning
SPARSITY_CONFIG.sparsity = 0.8 # target sparsity

Let's also write a utility to keep track of sparsity metrics.

In [5]:
def get_sparsity_metrics(params):
  sparsity_summary_dict = jaxpruner.utils.summarize_sparsity(
      param_tree=params, only_total_sparsity=False
  )
  return collections.OrderedDict(sparsity_summary_dict)

We create a `jaxpruner` updater from the given sparsity config. This is used to update the server optimization step to include pruning.

In [6]:
server_updater = jaxpruner.create_updater_from_config(SPARSITY_CONFIG)
server_gradient_transform = optax.adam(learning_rate=10**(-2.5), b1=0.9, b2=0.999, eps=10**(-4))
server_gradient_transform = server_updater.wrap_optax(server_gradient_transform)

### FedJAX Federated Averaging Implementation

This federated averaging implementation is forked from https://github.com/google/fedjax/blob/main/fedjax/algorithms/fed_avg.py, with minor changes to just two lines of code to incorporate `jaxpruner`.

In [7]:
ClientId = bytes
Grads = fedjax.Params


@fedjax.dataclass
class ServerState:
  """State of server passed between rounds.
  Attributes:
    params: A pytree representing the server model parameters.
    opt_state: A pytree representing the server optimizer state.
  """
  params: fedjax.Params
  opt_state: fedjax.OptState


def federated_averaging(
    grad_fn: Callable[[fedjax.Params, fedjax.BatchExample, fedjax.PRNGKey],
                      Grads],
    client_optimizer: fedjax.Optimizer,
    server_optimizer: fedjax.Optimizer,
    server_updater: jaxpruner.BaseUpdater,  # CHANGE #1: include the jaxpruner updater here!
    client_batch_hparams: fedjax.ShuffleRepeatBatchHParams
) -> fedjax.FederatedAlgorithm:
  """Builds the basic implementation of federated averaging."""

  def init(params: fedjax.Params) -> ServerState:
    opt_state = server_optimizer.init(params)
    return ServerState(params, opt_state)

  def apply(
      server_state: ServerState,
      clients: Sequence[Tuple[ClientId, fedjax.ClientDataset, fedjax.PRNGKey]]
  ) -> Tuple[ServerState, Mapping[ClientId, Any]]:
    client_diagnostics = {}
    # We use a list here for clarity, but we strongly recommend avoiding loading
    # all client outputs into memory since the outputs can be quite large
    # depending on the size of the model.
    client_delta_params_weights = []
    for client_id, client_dataset, client_rng in clients:
      delta_params = client_update(server_state.params, client_dataset,
                                   client_rng)
      client_delta_params_weights.append((delta_params, len(client_dataset)))
      # We record the l2 norm of client updates as an example, but it is not
      # required for the algorithm.
      client_diagnostics[client_id] = {
          'delta_l2_norm': fedjax.tree_util.tree_l2_norm(delta_params)
      }
    # The weighted mean is computed here and not with
    # fedjax.tree_util.tree_mean: in the fedjax 0.0.17 release that helper
    # calls jax.tree_map, which current JAX no longer has.
    total_weight = float(sum(w for _, w in client_delta_params_weights))
    mean_delta_params = jax.tree_util.tree_map(
        lambda *leaves: sum(
            leaf * (w / total_weight)
            for leaf, (_, w) in zip(leaves, client_delta_params_weights)),
        *[delta for delta, _ in client_delta_params_weights])
    server_state = server_update(server_state, mean_delta_params)
    return server_state, client_diagnostics

  def client_update(server_params, client_dataset, client_rng):
    params = server_params
    opt_state = client_optimizer.init(params)
    for batch in client_dataset.shuffle_repeat_batch(client_batch_hparams):
      client_rng, use_rng = jax.random.split(client_rng)
      grads = grad_fn(params, batch, use_rng)  # pytype: disable=wrong-arg-types  # jax-ndarray
      opt_state, params = client_optimizer.apply(grads, opt_state, params)
    delta_params = jax.tree_util.tree_map(lambda a, b: a - b,
                                               server_params, params)
    return delta_params

  def server_update(server_state, mean_delta_params):
    opt_state, params = server_optimizer.apply(mean_delta_params,
                                               server_state.opt_state,
                                               server_state.params)
    # CHANGE #2: apply the jaxpruner updater's post gradient update procedure
    # to the server model.
    updated_params = server_updater.post_gradient_update(params, opt_state)
    return ServerState(updated_params, opt_state)

  return fedjax.FederatedAlgorithm(init, apply)

### Federated Training


We are training a CNN model for character recognition on EMNIST, as in https://github.com/google/fedjax/blob/main/examples/emnist_fed_avg.py.

In [8]:
# Load train and test federated data for EMNIST.
train_fd, test_fd = fedjax.datasets.emnist.load_data(only_digits=False)

Reusing cached file '/home/dima/.cache/fedjax/federated_emnist_train.sqlite'


Reusing cached file '/home/dima/.cache/fedjax/federated_emnist_test.sqlite'


In [9]:
# Create CNN model with dropout.
model = fedjax.models.emnist.create_conv_model(only_digits=False)

In [10]:
# Scalar loss function with model parameters, batch of examples, and seed
# PRNGKey as input.
def loss(params, batch, rng):
  # `rng` used with `apply_for_train` to apply dropout during training.
  preds = model.apply_for_train(params, batch, rng)
  # Per example loss of shape [batch_size].
  example_loss = model.train_loss(batch, preds)
  return jnp.mean(example_loss)

In [11]:
# Gradient function of `loss` w.r.t. to model `params` (jitted for speed).
grad_fn = jax.jit(jax.grad(loss))

When instantiating the server optimizer, use the optax gradient transform produced by the `jaxpruner` updater that includes the pruning procedure.

In [12]:
# Create federated averaging algorithm, with server-side pruning.
client_optimizer = fedjax.optimizers.sgd(learning_rate=10**(-1.5))
server_optimizer = fedjax.optimizers.create_optimizer_from_optax(
    server_gradient_transform  # note this includes the pruning algorithm
)

Our federated averagining procedure, given above, applies a post gradient update pruning procedure to the server model, and requires the server updater as an argument.

In [13]:
# Hyperparameters for client local traing dataset preparation.
client_batch_hparams = fedjax.ShuffleRepeatBatchHParams(batch_size=20)
algorithm = federated_averaging(grad_fn, client_optimizer,
                                server_optimizer,
                                server_updater,  # note jaxpruner updater is a required arg
                                client_batch_hparams)

With pruning  integrated into the server optimizer and federated averaging algorithm, we can conduct federated training as usual.

In [14]:
# Initialize model parameters and algorithm server state.
init_params = model.init(jax.random.PRNGKey(17))
server_state = algorithm.init(init_params)

In [15]:
training_metrics_log = []
test_metrics_log = []
sparsity_metrics_log = []

# Train and eval loop.
train_client_sampler = fedjax.client_samplers.UniformGetClientSampler(
    fd=train_fd, num_clients=10, seed=0)
for round_num in range(1, 1501):
  # Sample 10 clients per round without replacement for training.
  clients = train_client_sampler.sample()
  # Run one round of training on sampled clients.
  server_state, client_diagnostics = algorithm.apply(server_state, clients)
  print(f'[round {round_num}]')
  # Optionally print client diagnostics if curious about each client's model
  # update's l2 norm.
  # print(f'[round {round_num}] client_diagnostics={client_diagnostics}')

  if round_num % 10 == 0:
    # Periodically evaluate the trained server model parameters.
    # Read and combine clients' train and test datasets for evaluation.
    client_ids = [cid for cid, _, _ in clients]
    train_eval_datasets = [cds for _, cds in train_fd.get_clients(client_ids)]
    test_eval_datasets = [cds for _, cds in test_fd.get_clients(client_ids)]
    train_eval_batches = fedjax.padded_batch_client_datasets(
        train_eval_datasets, batch_size=256)
    test_eval_batches = fedjax.padded_batch_client_datasets(
        test_eval_datasets, batch_size=256)

    # Run evaluation metrics defined in `model.eval_metrics`.
    train_metrics = fedjax.evaluate_model(model, server_state.params,  # pytype: disable=wrong-arg-types  # jax-ndarray
                                          train_eval_batches)
    training_metrics_log.append(train_metrics)
    test_metrics = fedjax.evaluate_model(model, server_state.params,  # pytype: disable=wrong-arg-types  # jax-ndarray
                                          test_eval_batches)
    sparsity_metrics = get_sparsity_metrics(server_state.params)
    sparsity_metrics_log.append(sparsity_metrics)
    test_metrics_log.append(test_metrics)
    print(f'[round {round_num}] train_metrics={train_metrics}')
    print(f'[round {round_num}] test_metrics={test_metrics}')
    print(f'[round {round_num}] sparsity_metrics={sparsity_metrics}')

# Save final trained model parameters to file.
fedjax.serialization.save_state(server_state.params, '/tmp/params')

W1012 18:23:56.119136  115671 cuda_timer.cc:88] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
W1012 18:23:56.284062  115671 cuda_timer.cc:88] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


W1012 18:23:56.434280  115671 cuda_timer.cc:88] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


[round 1]
[round 2]
[round 3]


[round 4]
[round 5]
[round 6]


[round 7]
[round 8]
[round 9]


...

[round 1491]
[round 1492]


[round 1493]
[round 1494]


[round 1495]
[round 1496]


[round 1497]
[round 1498]


[round 1499]
[round 1500]
[round 1500] train_metrics={'accuracy': Array(0.83874923, dtype=float32), 'loss': Array(0.43246293, dtype=float32)}
[round 1500] test_metrics={'accuracy': Array(0.82198954, dtype=float32), 'loss': Array(0.45894012, dtype=float32)}
[round 1500] sparsity_metrics=OrderedDict({'conv_dropout_module/conv2_d/b': Array(0., dtype=float32), 'conv_dropout_module/conv2_d/w': Array(0.7986111, dtype=float32), 'conv_dropout_module/conv2_d_1/b': Array(0., dtype=float32), 'conv_dropout_module/conv2_d_1/w': Array(0.8000217, dtype=float32), 'conv_dropout_module/linear/b': Array(0., dtype=float32), 'conv_dropout_module/linear/w': Array(0.79999965, dtype=float32), 'conv_dropout_module/linear_1/b': Array(0., dtype=float32), 'conv_dropout_module/linear_1/w': Array(0.8000252, dtype=float32), '_total_sparsity': Array(0.79981023, dtype=float32), '_nparams_nnz': Array(241547., dtype=float32), '_nparams': Array(1.20659e+06, dtype=float32)})


##  